# Vision Agents — 그림을 보는 에이전트 (데스크탑 Ollama판)

코스 섹션 [Vision and Browser agents](https://huggingface.co/learn/agents-course/unit2/smolagents/vision_agents)를 내 데스크탑 Ollama로 해 봐요.

**한 문장 요약: 글만 읽던 에이전트에게 "눈"(그림도 읽는 모델, VLM)을 달아 준다. 그림을 주는 방법은 두 가지 — ① 처음에 한꺼번에 주기 ② 일하는 중에 매 Step마다 화면을 찍어 넣어 주기(브라우저 에이전트)**

| 단계 | 하는 일 | 원문? | 필요한 것 |
|---|---|---|---|
| 0 | 준비 확인: **내 모델이 그림을 볼 수 있나?** | — | Ollama |
| 1 | 모델 연결 + 그림 한 장으로 "눈" 시험 | ★ 꼭 필요한 설정 하나 | 모델 |
| 2 | 1부: 처음에 그림 주기 — 조커인가 원더우먼인가 | 원문 | 모델 + 인터넷(그림 받기) |
| 3 | 2부: 브라우저 에이전트 — 일하는 중에 스크린샷 넣기 | 원문, **선택** | + 크롬, `helium`·`selenium`, 오래 걸림 |
| 4 | 원문 코드와 다른 점 (읽기만) | — | — |

**코스만 따라가려면 0 → 1 → 2** 가 핵심이고, 3은 하고 싶을 때 (크롬 창이 실제로 떠서 에이전트가 직접 클릭해요).

⚠️ 원문 경고: 이 예제는 **센 VLM**(GPT-4o)으로 시험했대요. 우리 9b 모델은 훨씬 작아서 설명이 틀리거나 브라우저 조작을 헤맬 수 있어요. **"구조가 어떻게 돌아가나"** 를 보는 게 목적이에요.

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 3단계용 **새 패키지 2개**(`helium`, `selenium`) → `git pull` 뒤 `pip install -r requirements.txt` 한 번.

## 0단계: 준비 확인 — 내 모델이 그림을 볼 수 있나?

Ollama에게 모델 정보(`/api/show`)를 물어보면 **할 수 있는 일 목록**(`capabilities`)을 알려 줘요. 여기에 **`vision`** 이 있어야 그림을 볼 수 있어요.

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

VISION_MODEL = "qwen3.5:9b"  # 0단계 결과가 False면 여기를 그림 보는 모델로 바꿔요 (아래 설명)


def ollama_post(path, body):
    req = urllib.request.Request(
        "http://localhost:11434" + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())


with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print(f"{VISION_MODEL} 있음?", VISION_MODEL in models)
caps = ollama_post("/api/show", {"model": VISION_MODEL}).get("capabilities", []) if VISION_MODEL in models else []
print("할 수 있는 일(capabilities):", caps)
print("👁️ 그림 볼 수 있음?", "vision" in caps)
for pkg in ["PIL", "requests", "helium", "selenium"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**
- `👁️ 그림 볼 수 있음? True` → 그대로 1단계로
- `False`면 지금 모델은 **글만** 읽어요 (`qwen3.5:9b`가 그림을 보는지는 미확인이라 이 칸으로 확인해요). 이럴 땐 그림 보는 모델을 하나 받아요. ⚠️ **약 6GB 다운로드**라 원할 때만, Jupyter를 끄고 PowerShell에서:
  ```
  ollama pull qwen2.5vl:7b
  ```
  받은 뒤 위 칸의 `VISION_MODEL = "qwen2.5vl:7b"`로 바꾸고 다시 실행
- `PIL`, `requests`는 `True`여야 해요 (smolagents와 같이 깔림). `helium`, `selenium`은 3단계(선택)용 → `False`면 `pip install -r requirements.txt`

## 1단계: 모델 연결 + "눈" 시험

지난 노트북들과 같은데 **한 줄이 늘었어요: `flatten_messages_as_text=False`** ★

- smolagents는 모델 이름이 `ollama`로 시작하면 기본으로 **"메시지를 글로만 납작하게 펴서"** 보내요 (`flatten_messages_as_text=True`). 그래서 지금까지 Ollama 노트북들은 그림 없이 잘 됐어요
- 그런데 그 상태로 그림을 주면 `Cannot use images with flatten_messages_as_text=True` 에러가 나요 (클라우드에서 smolagents 1.26으로 직접 확인)
- `False`로 끄면 그림이 Ollama까지 **그대로** 전달돼요 (원문은 GPT-4o라 이 문제가 없어요)

In [ ]:
from PIL import Image, ImageDraw
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id=f"ollama_chat/{VISION_MODEL}",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
    flatten_messages_as_text=False,  # ★ 그림을 보내려면 꼭 필요 (Ollama 모델은 기본이 True)
)

# 시험용 그림: 흰 바탕에 빨간 원 + 파란 네모 (인터넷 없이 직접 그림)
test_image = Image.new("RGB", (400, 300), "white")
draw = ImageDraw.Draw(test_image)
draw.ellipse((40, 60, 200, 220), fill="red")
draw.rectangle((240, 90, 360, 210), fill="blue")
display(test_image)

reply = model([
    {"role": "user", "content": [
        {"type": "text", "text": "What shapes and colors do you see? Answer in one short sentence."},
        {"type": "image", "image": test_image},  # 그림은 이렇게 글 옆에 같이 넣어요
    ]}
])
print("모델:", reply.content)

✅ **확인할 것**: 그림(빨간 원, 파란 네모)이 보이고, 모델이 **"red circle and blue square"** 비슷하게 답하면 눈이 달린 거예요
- 엉뚱한 답(그림을 못 본 듯)이면 → 0단계 `vision`이 `False`인 모델일 가능성
- 메시지 모양: `content`가 **목록**이고 그 안에 `{"type": "text", ...}`, `{"type": "image", ...}`가 나란히 → Unit 1 Messages 섹션의 "역할 + 내용"에서 내용 칸에 그림도 들어간 것

## 2단계 (원문 1부): 처음에 그림 주기 — 조커인가 원더우먼인가

상황: 원더우먼이라고 주장하는 손님이 왔는데, Alfred는 **조커가 변장한 것**이라고 의심해요. 예전 파티 손님 사진(여기선 위키백과 조커 사진 2장)을 에이전트에게 주고 확인시켜요.

먼저 원문 코드 그대로 그림을 받아요. `User-Agent` = "나는 이런 브라우저예요"라는 자기소개 글 (없으면 위키백과가 거절할 수 있음).

In [ ]:
from io import BytesIO

import requests

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",  # Joker image
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg",  # Joker image
]

images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    print("원래 크기:", image.size, end=" → ")
    image.thumbnail((768, 768))  # ★ 원문엔 없음: 긴 변을 768픽셀로 줄이기
    print("줄인 크기:", image.size)
    images.append(image)

for image in images:
    display(image)

✅ **확인할 것**: 조커 사진 2장 (밀랍 인형 + 만화 그림)

★ **왜 줄였나 (원문 밖, 짧게)**: 그림도 모델 안에서 **토큰**으로 바뀌어서 입력창을 차지해요. 그림이 클수록 토큰이 많아져요. 우리 입력창은 `num_ctx=8192` 토큰이라, 큰 사진 2장 + 에이전트 규칙서(약 2~3천 토큰)면 넘칠 수 있어요. GPT-4o는 입력창이 훨씬 커서 원문은 안 줄였어요.

이제 원문 에이전트예요. **도구가 하나도 없어요**(`tools=[]`). 그림은 `run(..., images=images)`로 질문과 **같이** 넣어요.

In [ ]:
from smolagents import CodeAgent

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,  # 원문: OpenAIServerModel(model_id="gpt-4o")
    max_steps=20,
    verbosity_level=2,
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images,
)
print("\n===== 최종 답 =====")
print(response)

✅ **확인할 것**
- 답에 **보라색 코트, 하얀 얼굴 분장, 빨간 입술, 초록 머리** 같은 말이 있고 "Joker"라고 했나? (원문 GPT-4o 결과도 이런 내용을 사전(dict) 모양으로 돌려줬어요. 우리 답 모양은 달라도 괜찮아요)
- 도구가 없는데 어떻게 그림을 "봤나"? → 에이전트가 따로 그림을 여는 게 아니라, 그림이 **질문과 같이 모델 입력**에 들어가서 모델이 직접 봐요. 에이전트는 본 내용을 `final_answer(...)`로 정리만 해요
- 틀리게 말하면(원더우먼이라고 하는 등) → 작은 모델의 한계. 원문도 "결과는 실행마다 달라질 수 있다"고 했어요

**(원문 밖·선택) 그림은 어디에 저장됐나?** 원문 2부에서 나오는 기억 칸 이름 중 하나가 여기서 바로 보여요.

In [ ]:
task_step = agent.memory.steps[0]
print("첫 번째 기억 칸:", type(task_step).__name__)
print("거기 붙은 그림 수(task_images):", len(task_step.task_images))

✅ `TaskStep`, `2` → 처음에 준 그림은 **TaskStep(질문 칸)의 `task_images`** 에 질문과 같이 저장되고, 매 Step 모델이 다시 봐요

## 3단계 (원문 2부, 선택): 브라우저 에이전트 — 일하는 중에 스크린샷 넣기

1부는 그림이 **처음부터 있을 때**예요. 손님이 사진첩에 없으면? → 에이전트가 **직접 웹을 돌아다니며** 화면을 보고 찾아야 해요.

원문의 핵심 그림 (에이전트 기억 = ReAct 기록장):

| 기억 칸 | 들어가는 것 | 그림은? |
|---|---|---|
| `SystemPromptStep` | 규칙서 (시스템 프롬프트) | — |
| `TaskStep` | 질문 + 처음 준 것 | `task_images` (2단계) |
| `ActionStep` (Step마다 1개) | 모델이 쓴 코드, 실행 결과(관찰) | **`observations_images`** ← 여기에 매 Step 브라우저 스크린샷 |

**새 부품 3개**
- **Selenium**: 파이썬으로 **진짜 브라우저(크롬)를 조종**하는 도구. `driver` = 조종간
- **Helium**: Selenium을 쉽게 쓰게 감싼 것. `go_to("주소")`, `click("글자")`, `scroll_down()` 처럼 사람 말에 가까운 명령
- **`step_callbacks`**: "Step이 끝날 때마다 이 함수를 불러 줘" 하고 걸어 두는 함수 (callback = 나중에 불러 달라고 맡겨 둔 함수). 여기선 `save_screenshot` → 매 Step 끝에 화면을 찍어 `observations_images`에 넣어요

**준비 (한 번만)**
1. `helium`, `selenium` 설치: `pip install -r requirements.txt` (원문은 `"smolagents[all]"`인데, 이건 큰 패키지를 잔뜩 받아서 필요한 2개만 넣었어요)
2. **크롬 브라우저**: PC에 크롬이 있으면 돼요. 크롬을 조종하는 작은 프로그램(chromedriver)은 Selenium이 **처음 실행할 때 알아서 받아요**

⚠️ 이 단계는 **크롬 창이 실제로 떠요.** 에이전트가 조종하는 동안 그 창을 건드리지 마세요. 9b 모델이라 **헤매거나 오래(10분+)** 걸릴 수 있어요. 멈추려면 Jupyter 위쪽 ■.

### 3-A: 크롬 띄우기

원문 본문엔 이 부분이 없고, 원문이 링크한 **Python 파일**(smolagents에 들어 있는 `vision_web_browser.py`)에 있어요. 창 크기를 1000×1350으로 고정해요.

In [ ]:
import helium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")

driver = helium.start_chrome(headless=False, options=chrome_options)  # headless=False → 창이 보여요
helium.go_to("https://en.wikipedia.org/wiki/Wonder_Woman")
print("지금 주소:", driver.current_url)

✅ **확인할 것**: 크롬 창이 하나 뜨고 위키백과 원더우먼 페이지가 열림
- `SessionNotCreatedException` / 크롬을 못 찾는다는 에러 → 크롬이 안 깔렸거나 버전 문제. 크롬을 설치·업데이트하고 다시

### 3-B: 브라우저용 도구 3개 (원문 그대로)

사람이 웹을 쓰는 동작을 도구로 만든 거예요: **Ctrl+F로 글자 찾기 / 뒤로 가기 / 팝업 닫기**. 셋 다 위에서 만든 `driver`(조종간)를 써요.

In [ ]:
from smolagents import tool


@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
    Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
    if nth_result > len(elements):
        raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
    result = f"Found {len(elements)} matches for '{text}'."
    elem = elements[nth_result - 1]
    driver.execute_script("arguments[0].scrollIntoView(true);", elem)
    result += f"Focused on element {nth_result} of {len(elements)}"
    return result


@tool
def go_back() -> None:
    """Goes back to previous page."""
    driver.back()


@tool
def close_popups() -> str:
    """
    Closes any visible modal or pop-up on the page. Use this to dismiss pop-up windows! This does not work on cookie consent banners.
    """
    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()


# 모델 없이 도구만 먼저: 지금 페이지에서 "tiara"(머리띠) 찾기
print(search_item_ctrl_f("tiara"))

✅ **확인할 것**: `Found N matches for 'tiara'.Focused on element 1 of N` + 크롬 창이 그 글자 있는 곳으로 **스크롤**됨

### 3-C: 매 Step 스크린샷 찍기 (`save_screenshot`)

⚠️ **원문 본문 코드는 smolagents 1.26에서 그대로 안 돼요** (클라우드에서 확인):
1. `agent.logs` → 이제 없는 이름이에요 (`AttributeError`). 지금은 `agent.memory.steps`
2. `isinstance(step_log, ...)`에서 변수 이름을 잘못 써서, **옛 스크린샷이 안 지워져요** → 그림이 쌓여 입력창이 금방 넘침
3. `step_logs.observations` 도 이름 오타

그래서 원문이 링크한 **Python 파일의 고친 버전**(1.26 안에 들어 있는 것)을 써요. 하는 일은 같아요:
- 1초 기다리기(화면 움직임이 끝나게) → **2 Step보다 오래된 스크린샷은 지우기**(입력창 아끼기) → 지금 화면을 찍어 이번 Step의 `observations_images`에 넣기 → 관찰 글에 지금 주소(URL)도 붙이기

In [ ]:
from time import sleep

from smolagents.agents import ActionStep


def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps:  # ★ 원문: agent.logs (1.26엔 없음)
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None  # 오래된 스크린샷 지우기
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        image.thumbnail((768, 768))  # ★ 원문엔 없음: 2단계처럼 작게 (입력창 아끼기)
        memory_step.observations_images = [image.copy()]  # Create a copy to ensure it persists, important!

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = (
        url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    )
    return

### 3-D: 브라우저 에이전트 실행 (원문)

원문 코드에 원문 본문에서 빠진 두 가지를 더했어요 (둘 다 원문이 링크한 Python 파일에 있음):
1. **`helium_instructions`**: 원문 질문 끝에 붙이는 "브라우저 사용 설명서"인데 본문엔 정의가 없어요. smolagents 안에 그대로 들어 있어서 가져와 써요. 아래 칸에서 앞부분을 출력해 봐요
2. **`agent.python_executor("from helium import *")`**: 에이전트의 파이썬 작업 공간에 `go_to`, `click` 같은 helium 명령을 **미리 꺼내 놓기**. 설명서에도 *"We've already ran from helium import \*"* 라고 써 있어요

In [ ]:
from smolagents import DuckDuckGoSearchTool
from smolagents.vision_web_browser import helium_instructions  # ★ 원문 본문엔 정의가 없음

print(helium_instructions[:600], "...\n")

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), go_back, close_popups, search_item_ctrl_f],
    model=model,  # 원문: OpenAIServerModel(model_id="gpt-4o")
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],  # ★ 매 Step 끝에 스크린샷
    max_steps=20,
    verbosity_level=2,
)
agent.python_executor("from helium import *")  # ★ 원문 본문엔 없음 (Python 파일엔 있음)

agent.run("""
I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.

Please search for images of Wonder Woman and generate a detailed visual description based on those images. Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can determine whether to grant her access to the event.
""" + helium_instructions)

✅ **확인할 것**
- 매 Step 끝에 `Captured a browser screenshot: (1000, 1350) pixels` 비슷한 줄 → 콜백(`save_screenshot`)이 돌았다는 뜻
- 모델 코드에 `go_to(...)`, `click(...)`, `scroll_down(...)` 같은 **helium 명령**이 나오고, 크롬 창이 **실제로 움직이나**?
- 관찰에 `Current url: ...` 이 붙어 있나? (콜백이 붙인 것)
- 원문 최종 답: *빨강·금색 상의, 별 무늬 파란 하의, 금색 티아라, 은색 팔찌, 금색 진실의 올가미, 본명 Diana Prince*. 내 답은 얼마나 비슷한가?
- 헤매면(같은 곳만 클릭, 로그인 시도, `LookupError` 반복) → 9b 모델로는 흔해요. 원문 Python 파일 주석에도 *"GPT-4o나 Claude-3.5-sonnet 정도여야 해낸다"* 고 써 있어요

다 봤으면 크롬을 닫아요.

In [ ]:
helium.kill_browser()
print("크롬 닫음")

## 4단계 (읽기만): 원문 코드와 다른 점

| 원문 | 이 노트북 | 왜 |
|---|---|---|
| `OpenAIServerModel("gpt-4o")` | Ollama `LiteLLMModel` + **`flatten_messages_as_text=False`** | 무료·내 PC. Ollama 모델은 기본이 "글로만 보내기"라 끄지 않으면 그림 에러 (1.26 확인) |
| 그림 원래 크기 | `thumbnail((768, 768))` | 그림도 토큰 → 우리 입력창 8192 |
| `save_screenshot`의 `agent.logs` 등 | 1.26 Python 파일 버전 (`agent.memory.steps`) | 원문 본문 코드는 1.26에서 에러 + 옛 스크린샷이 안 지워짐 |
| `helium_instructions` (본문에 정의 없음) | `from smolagents.vision_web_browser import helium_instructions` | smolagents 안에 들어 있음 |
| 드라이버 시작, `from helium import *` (본문에 없음) | 3-A, 3-D에 넣음 | Python 파일에 있는 준비 단계 |
| `pip install "smolagents[all]" helium selenium` | `helium`, `selenium`만 | `[all]`은 무거운 패키지를 많이 받음 |

**끝!** 결과(1단계 모델 답, 2단계 최종 답, 3단계를 했다면 Step 수·모델이 쓴 helium 명령·최종 답)를 복사해서 AI에게 붙여 주면 같이 분석해요.
Jupyter 끄기: `Esc` → `Ctrl + C` 두 번.